# Ajuste de hiperparâmetros do modelo candidato

Este notebook ajusta somente o `hist_gradient_boosting`, candidato vencedor da comparação baseline. O ajuste usa previsões out-of-fold dentro do bloco `fit`, sem threshold, sem calibração e sem consultar o teste oficial.

## 1. Protocolo do ajuste

A pergunta é se uma configuração diferente dos hiperparâmetros melhora a qualidade das probabilidades antes de calibrar e escolher o threshold. A busca é uma grade pequena e explícita, comparada por PR-AUC, ROC-AUC e Brier em OOF group-aware.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from adult_data import load_adult
from hyperparameter_tuning import write_hgb_tuning_outputs

data = load_adult(ROOT / 'data' / 'raw' / 'adult_uci')
tuning_dir = ROOT / 'data' / 'processed' / 'tuning'
outputs = write_hgb_tuning_outputs(data, tuning_dir)
outputs

{'metrics': WindowsPath('D:/Projetos/ModelRisk_ResponsibleML/data/processed/tuning/hgb_tuning_metrics.csv'),
 'contract': WindowsPath('D:/Projetos/ModelRisk_ResponsibleML/data/processed/tuning/hgb_tuning_contract.json')}

## 2. Resultados OOF

As métricas abaixo avaliam as probabilidades out-of-fold no bloco `fit`. Nenhuma delas depende de escolher um threshold operacional; essa decisão continua reservada ao Notebook 5, depois da calibração.

In [2]:
metrics = pd.read_csv(tuning_dir / 'hgb_tuning_metrics.csv')
contract = json.loads((tuning_dir / 'hgb_tuning_contract.json').read_text(encoding='utf-8'))
display(metrics.round(4))
display(pd.Series(contract, name='valor').to_frame())

,candidate_id,max_iter,learning_rate,max_leaf_nodes,min_samples_leaf,roc_auc,pr_auc,brier
0,conservative_leaves,250,0.05,15,30,0.9233,0.8188,0.0910
1,baseline_fixed,200,0.08,31,20,0.9234,0.8184,0.0910
2,deeper_regularized,300,0.05,31,50,0.9226,0.8176,0.0914
3,shallow_regularized,300,0.08,15,50,0.9228,0.8171,0.0914
4,faster_medium,150,0.10,31,20,0.9227,0.8166,0.0915
5,wide_faster,200,0.12,63,20,0.9216,0.8149,0.0923


,valor
protocol_version,hgb-tuning-mvp-1
tuned_model,hist_gradient_boosting
search_method,explicit_candidate_grid
candidate_count,6
candidate_ids,"[baseline_fixed, conservative_leaves, deeper_r..."
objective_order,"[pr_auc, roc_auc, brier]"
fit_rows,19537
oof_folds,3
group_aware,True
random_state,44


### Insights:
A configuração vencedora é registrada pelo maior PR-AUC OOF, usando ROC-AUC e Brier como critérios de desempate. O threshold ainda não participa dessa escolha. O resultado deve ser interpretado como a melhor configuração dentro da grade candidata, não como uma busca exaustiva de todos os hiperparâmetros possíveis.

## 3. Encaminhamento para a seleção final

O contrato de tuning fornece os hiperparâmetros selecionados para o Notebook 5. Lá, o pipeline será ajustado no bloco `fit`, calibrado no bloco `calibration` e avaliado na grade de thresholds do bloco `selection`. O `test_official` continua reservado.